In [ ]:
import torch
from torch import nn

In [ ]:
if torch.cuda.is_available():
     device = "cuda"
elif torch.backends.mps.is_available():
    device = "mps"
else:
    device = "cpu"

In [ ]:
import numpy as np
import torch
from sklearn.model_selection import train_test_split

X = np.arange(0, 1, 0.02).reshape(-1, 1)
y = (5.66 * X + 3.227).reshape(-1, 1)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

X_train = torch.from_numpy(X_train).float()
X_test = torch.from_numpy(X_test).float()
y_train = torch.from_numpy(y_train).float()
y_test = torch.from_numpy(y_test).float()

In [ ]:
class LinearRegressionNetwork(nn.Module):   
    def __init__(self):
        super().__init__()
        self.weights = nn.Parameter(torch.randn(1, requires_grad=True,dtype=torch.float))
        self.bias = nn.Parameter(torch.randn(1, requires_grad=True,dtype=torch.float))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.weights*x + self.bias


In [ ]:
torch.manual_seed(42)
model = LinearRegressionNetwork()
model.state_dict()

In [ ]:
import matplotlib.pyplot as plt 

def plot_data(X_train, y_train, X_test, y_test, predictions=None):
    plt.figure(figsize=(8, 5))

    plt.scatter(X_train, y_train, color="blue", label="Train")
    plt.scatter(X_test, y_test, color="green", label="Test")

    if predictions is not None:
        plt.scatter(X_test, predictions, color="red", label="Predictions")

    plt.legend()
    plt.show()

In [ ]:
with torch.inference_mode():
    y_pred = model(X_test)

plot_data(X_train, y_train, X_test, y_test, predictions=y_pred)

In [ ]:
loss_fn = nn.MSELoss()
optimizer = torch.optim.SGD(params=model.parameters(), lr=0.1)

In [ ]:
epochs = 200
train_loss_list, test_loss_list = [], []

for epoch in range(epochs):
    model.train()
    train_pred = model(X_train)
    loss = loss_fn(y_train, train_pred)
    optimizer.zero_grad()
    loss.backward()
    model.eval()
    with torch.inference_mode():
        test_pred = model(X_test)
        test_loss = loss_fn(y_test, test_pred)
    train_loss_list.append(loss.detach().numpy())
    test_loss_list.append(test_loss.detach().numpy())


In [ ]:
with torch.inference_mode():
    y_pred = model(X_test)

plot_data(
    X_train=X_train,
    y_train=y_train,
    X_test=X_test,
    y_test=y_test,
    predictions=y_pred
)

In [ ]:
plt.plot(range(epochs), train_loss_list, label="Training Loss")
plt.plot(range(epochs), test_loss_list, label="Testing Loss")
plt.title("Epochs vs Loss");
plt.xlabel("Epochs");
plt.ylabel("Loss")
plt.legend();
plt.show()